In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\deforestation_detection\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\deforestation_detection'

In [6]:
#1. Entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    data_file:Path
    STATUS_FILE: Path

In [7]:
from deforestation_detection.constant import *
from deforestation_detection.utils.common import *

In [8]:
#2. Configuration Manager

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            data_file= config.data_file,
            STATUS_FILE = config.STATUS_FILE
        )

        return data_validation_config

In [10]:
from deforestation_detection.logging import logger
import torch

In [11]:
#3. components

class DataValidation:
    def __init__(self, config):
        self.config = config

    def validate_tensor_data(self) -> bool:
        try:
            validation_status = True
            
            #file exist or not
            if not os.path.exists(self.config.data_file):
                validation_status = False
                logger.error(f"Data file not found at {self.config.data_file}")
            else:
                logger.info(f"Loading dataset from {self.config.data_file} for validation...")
                data = torch.load(self.config.data_file)
                
                #check values based on structure
                if isinstance(data, dict):
                    for key, tensor in data.items():
                        if torch.isnan(tensor).any() or torch.isinf(tensor).any():
                            logger.error(f"NaN or Inf values detected in tensor key: {key}")
                            validation_status = False
                        else:
                            logger.info(f"Tensor key '{key}' validated successfully with shape: {tensor.shape}")
                            
                elif isinstance(data, (list, tuple)):
                    for idx, tensor in enumerate(data):
                        if torch.isnan(tensor).any() or torch.isinf(tensor).any():
                            logger.error(f"NaN or Inf values detected at tensor index {idx}")
                            validation_status = False
                        else:
                            logger.info(f"Tensor at index {idx} validated successfully with shape: {tensor.shape}")
                            
                else:
                    #single tensor case
                    if torch.isnan(data).any() or torch.isinf(data).any():
                        logger.error("NaN or Inf values detected in dataset tensor.")
                        validation_status = False
                    else:
                        logger.info(f"Dataset tensor validated successfully with shape: {data.shape}")

            #status file writing
            os.makedirs(Path(self.config.STATUS_FILE).parent, exist_ok=True)
            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation status: {validation_status}")
                
            logger.info(f"Validation status recorded as: {validation_status} at {self.config.STATUS_FILE}")
            return validation_status

        except Exception as e:
            raise e

In [14]:
#4. pipeline

try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config = data_validation_config)
    data_validation.validate_tensor_data()

except Exception as e:
    raise e

[2026-10-21 20:34:19,090: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-10-21 20:34:19,093: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-10-21 20:34:19,095: INFO: common: created directory at artifacts]
[2026-10-21 20:34:19,097: INFO: common: created directory at artifacts/data_validation]
[2026-10-21 20:34:19,098: INFO: 2415517506: Loading dataset from artifacts/data_ingestion/dataset_arrays.pt for validation...]
[2026-10-21 20:34:19,276: INFO: 2415517506: Tensor key 'X_full' validated successfully with shape: torch.Size([144, 4, 256, 256])]
[2026-10-21 20:34:19,293: INFO: 2415517506: Tensor key 'y_full' validated successfully with shape: torch.Size([144, 1, 256, 256])]
[2026-10-21 20:34:19,296: INFO: 2415517506: Validation status recorded as: True at artifacts/data_validation/status.txt]
